# Dashboard & ML POC demo

This notebook demonstrates loading curated sample datasets, validating ingestion, running rule-based detectors, visualizing detections on candlestick charts, performing a parameter sweep to calibrate detectors, and a quick JupyterDash demo for the dashboard.

Open the notebook and run each cell sequentially. The notebook saves sample CSVs and detection artifacts under `data/samples/` and `artifacts/` respectively.

In [ ]:
## 1) Install dev requirements & verify kernel

# NOTE: Running these installs inside a shared environment may change your local env. If you prefer, run these commands in your terminal instead.
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]", "jupyter-dash", "plotly", "pytest"])
print('Installed dev extras and notebook deps (if not already present)')

In [ ]:
## 2) Load or generate curated sample datasets

import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ensure data dir exists
os.makedirs('data/samples', exist_ok=True)

# generate a synthetic OHLCV series with a few injected patterns
def generate_synthetic_ohlcv(start='2026-01-01', n=120, seed=42):
    rng=np.random.RandomState(seed)
    t0=datetime.fromisoformat(start)
    ts=[(t0+timedelta(minutes=15*i)).isoformat() for i in range(n)]
    price=100+np.cumsum(rng.randn(n)*0.5)
    o=price + rng.randn(n)*0.05
    c=price + rng.randn(n)*0.05
    h=np.maximum(o,c) + np.abs(rng.randn(n)*0.2)
    l=np.minimum(o,c) - np.abs(rng.randn(n)*0.2)
    v=(rng.randint(100,1000,size=n)).astype(int)
    df=pd.DataFrame({'timestamp':ts,'open':o,'high':h,'low':l,'close':c,'volume':v})
    return df

sdf=generate_synthetic_ohlcv()
sdf.to_csv('data/samples/sample_synthetic.csv', index=False)
print('Wrote sample dataset to data/samples/sample_synthetic.csv')

# create a labeled sample by injecting an artificial 'doji' style candle and a bullish engulfing
ldf=sdf.copy()
# inject doji at index 20
ldf.loc[20,'open']=100.0
ldf.loc[20,'close']=100.02
# inject bullish engulfing at 50-51
ldf.loc[50,'open']=102.0; ldf.loc[50,'close']=101.5
ldf.loc[51,'open']=101.0; ldf.loc[51,'close']=103.5
ldf.to_csv('data/samples/sample_labeled.csv', index=False)
print('Wrote labeled sample dataset to data/samples/sample_labeled.csv')

In [ ]:
## 3) CSV ingestion & validation check

from candle_patterns.ingestion import load_csv, validate_schema

for p in ['data/samples/sample_synthetic.csv','data/samples/sample_labeled.csv']:
    print('\n--',p)
    df=load_csv(p)
    print(df.dtypes)
    errors=validate_schema(df)
    if errors:
        print('Validation errors:', errors)
    else:
        print('Validation OK')

In [ ]:
## 4) Run rule-based detector on samples

from candle_patterns.detection import detect_patterns

# run detection and save a small report
os.makedirs('artifacts/detections', exist_ok=True)

for p in ['data/samples/sample_synthetic.csv','data/samples/sample_labeled.csv']:
    df=load_csv(p)
    dets=detect_patterns(df)
    print(p, '->', len(dets), 'detections')
    out='artifacts/detections/' + os.path.basename(p).replace('.csv','_detections.csv')
    pd.DataFrame(dets).to_csv(out, index=False)
    print('Saved detections to', out)


In [ ]:
## 5) Visualize detections on candlestick charts

import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_with_annotations(df, detections, title='Candles with annotations'):
    fig = make_subplots(rows=1, cols=1)
    fig.add_trace(go.Candlestick(x=df['timestamp'], open=df['open'], high=df['high'], low=df['low'], close=df['close'], name='OHLC'))
    # add markers for detections
    for d in detections:
        ts=d.get('timestamp')
        pat=d.get('pattern')
        fig.add_trace(go.Scatter(x=[ts], y=[df.loc[df['timestamp']==ts,'high'].iat[0]], mode='markers+text', text=[pat], textposition='top center', marker=dict(size=10)))
    fig.update_layout(title=title, xaxis_rangeslider_visible=False, height=600)
    return fig

# show a sample
sdf=load_csv('data/samples/sample_labeled.csv')
dets=pd.read_csv('artifacts/detections/sample_labeled_detections.csv') if os.path.exists('artifacts/detections/sample_labeled_detections.csv') else pd.DataFrame(detect_patterns(sdf))
fig=plot_with_annotations(sdf, dets.to_dict('records'), title='Sample labeled dataset')
fig.show()
# save an interactive HTML snapshot
os.makedirs('artifacts/plots', exist_ok=True)
fig.write_html('artifacts/plots/sample_labeled_plot.html')
print('Saved interactive plot to artifacts/plots/sample_labeled_plot.html')

In [ ]:
## 6) Calibrate detectors (parameter sweep & metrics)

from sklearn.metrics import precision_recall_fscore_support

# for this demo, assume 'sample_labeled.csv' has injected pattern ground-truth at known timestamps
# Make a simple sweep where we change a hypothetical 'body_ratio' threshold to filter detections

def evaluate_sweep(df, true_detections_ts, body_thresholds=[0.01,0.02,0.05,0.1]):
    rows=[]
    for t in body_thresholds:
        # run detector and filter by synthetic 'body' size metric
        dets=detect_patterns(df)
        # compute body size as abs(close-open)
        dets_filtered=[d for d in dets if abs(df.loc[df['timestamp']==d['timestamp'],'close'].iat[0]-df.loc[df['timestamp']==d['timestamp'],'open'].iat[0]) >= t]
        pred_ts=set([d['timestamp'] for d in dets_filtered])
        y_true=[1 if ts in true_detections_ts else 0 for ts in df['timestamp']]
        y_pred=[1 if ts in pred_ts else 0 for ts in df['timestamp']]
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        rows.append({'threshold':t,'precision':precision,'recall':recall,'f1':f1})
    return pd.DataFrame(rows)

# for the demo, create a truth set from our injected labeled file at index 20 and 50/51
truth_ts={sdf['timestamp'].iat[20], sdf['timestamp'].iat[51]}
results=evaluate_sweep(sdf, truth_ts)
print(results)
results.plot(x='threshold', y=['precision','recall','f1'], kind='line', marker='o')

In [ ]:
## 7) Integration test harness (CLI + dataset)

import subprocess
import shlex

# run the CLI analyzer against a sample file and assert output
os.makedirs('artifacts/cli_reports', exist_ok=True)
out_file='artifacts/cli_reports/sample_report.csv'
cmd=f"python -m candle_patterns.cli run data/samples/sample_labeled.csv --out {out_file}"
print('Running:',cmd)
res=subprocess.run(shlex.split(cmd), capture_output=True, text=True)
print('Return code:', res.returncode)
print('STDOUT:', res.stdout)
print('STDERR:', res.stderr)
assert res.returncode==0, 'CLI failed'
assert os.path.exists(out_file), 'Expected report not created'
print('CLI smoke test OK, report at', out_file)


In [ ]:
## 8) Dashboard quick demo (JupyterDash embed)

# A minimal inline JupyterDash demo (works if jupyter-dash is installed).
# This is a lightweight demo and not a replacement for the full Dash server.

try:
    from jupyter_dash import JupyterDash
    from dash import html, dcc
    import plotly

    app = JupyterDash(__name__)
    app.layout = html.Div([
        html.H4('Notebook: Dashboard quick demo'),
        dcc.Graph(figure=fig),
    ])

    print('Starting JupyterDash in inline mode (press stop to continue).')
    app.run_server(mode='inline', port=8051)
except Exception as e:
    print('JupyterDash inline demo requires jupyter-dash; run the cell in a Jupyter environment to demo the app. Error:', e)

print('\nFull Dash app is in src/candle_patterns/dashboard.py. Run `python -m dash` in the project root to start the full app.')

## 9) Save artifacts & PR checklist

# Save chosen parameters or calibration defaults
os.makedirs('artifacts/config', exist_ok=True)
calib_defaults={'body_threshold':0.02}
import json
with open('artifacts/config/calibrated_detector_params.json','w') as fh:
    json.dump(calib_defaults, fh)
print('Saved calibrated params to artifacts/config/calibrated_detector_params.json')

# Checklist for PR:
# - Add curated CSVs to data/samples/ (done)
# - Add unit/integration tests for the new samples and CLI (add tests under tests/)
# - Include notebooks and artifacts in PR where appropriate
# - Ensure linter and tests pass locally: `ruff check .`, `black --check .`, `pytest -q`

print('\nPR checklist: The repo contains sample data, demo notebooks, and artifact outputs. Update the tracker row and target PR to `feature/mvp-setup`.')